In [1]:
%cd ..

/Users/lallannkhann/Documents/AIC26/backend


/opt/miniconda3/envs/dezus_py310/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [ ]:
import json
import numpy as np 

from src.core.config import load_settings
from src.models import Video, Scene, Shot, Keyframe
from src.database import MilvusDatabase, MongoDatabase, ElasticDatabase
from src.database.utils import pydantic_to_milvus_schema, pydantic_to_mongo_schema, pydantic_to_elastic_schema

In [3]:
settings = load_settings()

In [4]:
elastic_db = ElasticDatabase(settings.elastic)
keyframe_schema = pydantic_to_elastic_schema(Keyframe)

elastic_db.create_collection(
    collection_name="keyframe",
    schema=keyframe_schema
)

elastic_db.get_collection(collection_name="keyframe")

keyframe_data = {
    "id": "L01_V001_S001_SH001_K001",
    "path": "/resource/keyframes/L01_V001_S001_SH001_K001.jpg",
    "timestamp": 12.34,
    "description": "A slide showing the introduction to machine learning",
    "ocr": ["Chapter 1: Introduction", "Machine Learning Basics"],
    "is_deleted": False,
    "is_processed":True
}

elastic_db.index_data(
    collection_name="keyframe",
    data=keyframe_data,
    data_id=keyframe_data["id"]
)

keyframe_data["timestamp"] = 11
elastic_db.update_data(
    collection_name="keyframe",
    data=keyframe_data,
    data_id=keyframe_data["id"]
)

results = elastic_db.search(
    collection_name="keyframe",
    query={
        "match": {
            "ocr": "Machine Learning"
        }
    }
)
print(results)

# elastic_db.delete_data(
#     collection_name="keyframe",
#     data_id=keyframe_data["id"]
# )

# elastic_db.delete_collection(
#     collection_name="keyframe"
# )

{'took': 1, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 0, 'relation': 'eq'}, 'max_score': None, 'hits': []}}


In [ ]:
mongo_db = MongoDatabase(settings.mongo)
video_schema = pydantic_to_mongo_schema(Video)

mongo_db.create_collection(
    collection_name="video",
    schema=video_schema
)

mongo_db.get_collection(collection_name="video")

video_data = {
    "id": "L001_V001",
    "path": "/resource/videos/L001_V001.mp4",
    "fps": 30,
    "metadata": {
        "codec": "h264",
        "bitrate": "5000k"
    },
    "is_deleted": False,
    "is_processed": True
}

mongo_db.index_data(
    collection_name="video",
    data=video_data,
    data_id=video_data["id"]
)

mongo_db.update_data(
    collection_name="video",
    data={"fps": 25},
    data_id=video_data["id"]
)

results = mongo_db.search(
    collection_name="video",
    query={}
)
print(results)

# mongo_db.delete_data(
#     collection_name="video",
#     data_id="L001_V001"
# )

# mongo_db.delete_collection(
#     collection_name="video"
# )

In [ ]:
milvus_db = MilvusDatabase(settings.milvus)
video_schema = pydantic_to_milvus_schema(
    model=Video,
    primary_key="id",
    embedding_field="embedding",
    embedding_dim=512
)

index_params = {
    "index_type": "IVF_FLAT",
    "metric_type": "L2",
    "params": {"nlist": 128}
}

milvus_db.create_collection(
    collection_name="video_clip",
    schema=video_schema,
    index_params=index_params
)

milvus_db.get_collection(collection_name="video_clip")

embedding_dim = 512
fake_embedding = np.random.rand(embedding_dim).astype(float).tolist()

video_data = {
    "id": "L001_V001",
    "path": "/resource/videos/L001_V001.mp4",
    "fps": 30,
    "metadata": json.dumps({
        "codec": "h264",
        "bitrate": "5000k"
    }),
    "is_deleted": False,
    "is_processed": True,
    "embedding": fake_embedding
}

milvus_db.index_data(
    collection_name="video_clip",
    data=video_data,
    data_id=video_data["id"]
)

video_data["fps"] = 25

milvus_db.update_data(
    collection_name="video_clip",
    data=video_data,
    data_id=video_data["id"]
)

results = milvus_db.search(
    collection_name="video_clip",
    query={
        "data": [fake_embedding],
        "param": {
            "metric_type": "L2",
            "params": {"nprobe": 10}
        },
        "limit": 1,
        "output_fields": ["id", "path", "fps", "metadata"],
        "index_params": {
            "index_type": "IVF_FLAT",
            "metric_type": "L2",
            "params": {"nlist": 128}
        }
    }
)
for hits in results:
    for hit in hits:
        print(f"ID: {hit.id}, distance: {hit.distance}")

# milvus_db.delete_data(
#     collection_name="video_clip",
#     data_id="L001_V001"
# )

# milvus_db.delete_collection(
#     collection_name="video_clip"
# )

In [7]:
elastic_db = ElasticDatabase(settings.elastic)
video_schema = pydantic_to_elastic_schema(Video)

elastic_db.create_collection(
    collection_name="video",
    schema=video_schema
)

elastic_db.get_collection(collection_name="video")

video_data = {
    "id": "L001_V001",
    "path": "/resource/videos/L001_V001.mp4",
    "fps": 30,
    "metadata": {
        "codec": "h264",
        "bitrate": "5000k"
    },
    "is_deleted": False,
    "is_processed": True
}

elastic_db.index_data(
    collection_name="video",
    data=video_data,
    data_id=video_data["id"]
)

import time
time.sleep(1) # wait for elastic to index

elastic_db.update_data(
    collection_name="video",
    data={"doc": {"fps": 25}},
    data_id=video_data["id"]
)

time.sleep(1)

results = elastic_db.search(
    collection_name="video",
    query={"match_all": {}}
)
print(results)


{'took': 3, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 1, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'video', '_id': 'L001_V001', '_score': 1.0, '_source': {'id': 'L001_V001', 'path': '/resource/videos/L001_V001.mp4', 'fps': 30, 'metadata': {'codec': 'h264', 'bitrate': '5000k'}, 'is_deleted': False, 'is_processed': True, 'doc': {'fps': 25}}}]}}
